# Population exposed within distance bands downstream of PDGLs
This notebook computes population exposed within **cumulative distance bands** (e.g., 5, 10, 20 km) **downstream along a modeled flow path** for each PDGL, using a population raster (e.g., WorldPop).

You can run this in two modes:
1) **Flow-path mode (recommended)**: you have a line (polyline) flow path per lake (or per outlet), and we buffer a downstream segment of length *d*.
2) **Simple buffer mode**: if you do not have flow paths, we approximate exposure using circular buffers around the lake (less accurate for “downstream”).

Outputs:
- Per-PDGL population within 5/10/20 km (cumulative)
- Totals summed across all PDGLs
- Optional: a GeoPackage with the generated exposure polygons


In [1]:
# If running on Colab, uncomment:
# !pip -q install geopandas rasterio rioxarray shapely pyproj pandas numpy tqdm

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm.auto import tqdm

import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.io import MemoryFile
from rasterio.mask import mask

from shapely.geometry import mapping
from shapely.ops import substring

print("Imports OK")


Imports OK


## 1) Inputs
Set paths to:
- Your **HKH-2025 lake dataset** (GeoPackage/Shapefile/GeoJSON)
- Your **WorldPop population raster** (GeoTIFF)
- (Optional) **flow paths** (lines) with a lake id column matching the lakes dataset


In [2]:
# === User inputs ===
LAKES_PATH = r"D:/Thesis/glacial_lake_detection_thesis/Inference/5 Stats/lakes_2025_PDGL.gpkg"      # e.g., ".../hkh-2025-lakes.gpkg"
LAKES_LAYER = None                               # set to layer name if GPKG has multiple layers

FLOW_PATHS_PATH = None     # optional; set to None if not available
FLOW_LAYER = None                                # optional layer name
FLOW_LAKE_ID_COL = "lake_id"                     # id column in flow paths

POP_RASTER_PATH = r"D:/Thesis/glacial_lake_detection_thesis/Inference/5 Stats/inputs/global_pop_2025_CN_1km_R2025A_UA_v1.tif"

# Columns in lakes dataset
LAKE_ID_COL = "lake_id"                          # unique id per lake (must match flow paths if used)
PDGL_COL = "is_PDGL"                                # True/False flag column you added

# Distance bands in km (cumulative)
DISTANCE_BANDS_KM = [5, 10, 20]

# Corridor buffer width around flow path segment (meters).
# Think of this as a "valley buffer" or inundation proxy width.
VALLEY_BUFFER_M = 500  # adjust (e.g., 200, 500, 1000)

# If you do not have flow paths, set USE_FLOW_PATHS=False to use simple circular buffers (less accurate)
USE_FLOW_PATHS = False


## 2) Load data

In [3]:
def read_vector(path, layer=None):
    if layer is None:
        return gpd.read_file(path)
    return gpd.read_file(path, layer=layer)

lakes = read_vector(LAKES_PATH, LAKES_LAYER)
assert PDGL_COL in lakes.columns, f"Missing PDGL column: {PDGL_COL}"
assert LAKE_ID_COL in lakes.columns, f"Missing lake id column: {LAKE_ID_COL}"

# Keep only PDGLs
pdgl = lakes[lakes[PDGL_COL].astype(bool)].copy()
pdgl = pdgl[~pdgl.geometry.is_empty & pdgl.geometry.notna()].copy()

print("Total lakes:", len(lakes))
print("PDGL lakes:", len(pdgl))
pdgl.head(3)


Total lakes: 15353
PDGL lakes: 4519


,source_img,area_m2,lake_id,area_km2,perimeter_km,parent_glacier_idx,G11_mean_glacier_slope_deg,R1_steep_nonglac_km2,D3_dam_mean_slope_deg,H7_watershed_km2,...,L9,G11_norm,R1_norm,D3_norm,H7_norm,L9_norm,hazard_index,hazard_class,is_PDGL,geometry
19,20250902_060832_78_24da_3B_Visual_clip_reproje...,137.367649,20,0.000137,0.046211,356880.0,30.475851,4.030429,36.980495,NaN,...,0.046211,0.475027,0.191289,0.411069,NaN,0.004019,0.262806,high,True,"POLYGON ((7468201.926 4220920.88, 7468205.645 ..."
30,20250902_060832_78_24da_3B_Visual_clip_reproje...,131.741643,31,0.000132,0.045587,363571.0,38.859287,6.718119,36.772396,NaN,...,0.045587,0.618172,0.318849,0.408754,NaN,0.003949,0.332852,very_high,True,"POLYGON ((7461838.578 4225545.552, 7461838.752..."
31,20250902_060832_78_24da_3B_Visual_clip_reproje...,126.603402,32,0.000127,0.042660,329001.0,37.124931,0.299041,31.107368,NaN,...,0.042660,0.588558,0.014193,0.345740,NaN,0.003626,0.235427,high,True,"POLYGON ((7461161.991 4229181.55, 7461160.612 ..."


## 3) Choose a metric CRS (meters)
We need meters for 5/10/20 km distances. We'll reproject everything to an estimated UTM CRS based on your PDGL geometries.

In [4]:
# Estimate a good local projected CRS (UTM) from the PDGL geometries
metric_crs = pdgl.estimate_utm_crs()
print("Using metric CRS:", metric_crs)

pdgl_m = pdgl.to_crs(metric_crs)

# Load flow paths if used
flow = None
if USE_FLOW_PATHS:
    if FLOW_PATHS_PATH is None or (isinstance(FLOW_PATHS_PATH, str) and FLOW_PATHS_PATH.strip().lower() == "none"):
        raise ValueError("USE_FLOW_PATHS=True but FLOW_PATHS_PATH is None.")
    flow = read_vector(FLOW_PATHS_PATH, FLOW_LAYER)
    assert FLOW_LAKE_ID_COL in flow.columns, f"Missing flow lake id column: {FLOW_LAKE_ID_COL}"
    flow = flow[~flow.geometry.is_empty & flow.geometry.notna()].copy()
    flow_m = flow.to_crs(metric_crs)
else:
    flow_m = None

print("Flow paths loaded:" , (flow_m is not None))


Using metric CRS: EPSG:32643
Flow paths loaded: False


## 4) Build downstream exposure polygons
### Flow-path mode
For each PDGL we:
1) Find its corresponding flow line
2) Take the downstream segment from distance 0 to *d*
3) Buffer the segment by `VALLEY_BUFFER_M`

**Important:** This assumes each flow line starts at the lake/outlet and continues downstream. If your lines are reversed, reverse them before running (or see helper below).

In [5]:
def maybe_reverse_line(line, reverse=False):
    if not reverse:
        return line
    # reverse by swapping coordinates
    coords = list(line.coords)[::-1]
    from shapely.geometry import LineString
    return LineString(coords)

def downstream_corridor_from_line(line, distance_m, buffer_m):
    """Return buffered polygon of the line segment from 0..distance_m (cumulative)."""
    if line is None or line.is_empty:
        return None
    seg_len = min(distance_m, line.length)
    seg = substring(line, 0, seg_len)  # Shapely 2: takes start/end distances along line
    return seg.buffer(buffer_m)

def build_exposure_polygons(pdgl_m, distances_km, buffer_m, flow_m=None, id_col="lake_id", flow_id_col="lake_id"):
    distances_m = [int(d * 1000) for d in distances_km]
    records = []

    # Build a lookup for flow lines by lake_id (supports duplicates by taking the first)
    flow_lookup = {}
    if flow_m is not None:
        for _, r in flow_m.iterrows():
            lid = r[flow_id_col]
            if lid not in flow_lookup:
                flow_lookup[lid] = r.geometry

    for _, r in tqdm(pdgl_m.iterrows(), total=len(pdgl_m)):
        lid = r[id_col]
        geom = r.geometry

        for d_km, d_m in zip(distances_km, distances_m):
            if flow_m is not None:
                line = flow_lookup.get(lid, None)
                poly = downstream_corridor_from_line(line, d_m, buffer_m) if line is not None else None
            else:
                # Simple circular buffer around lake geometry (centroid) - fallback
                poly = geom.centroid.buffer(d_m)

            if poly is None or poly.is_empty:
                continue

            records.append({
                id_col: lid,
                "distance_km": d_km,
                "geometry": poly
            })

    return gpd.GeoDataFrame(records, crs=pdgl_m.crs)

exposure_polys = build_exposure_polygons(
    pdgl_m=pdgl_m,
    distances_km=DISTANCE_BANDS_KM,
    buffer_m=VALLEY_BUFFER_M,
    flow_m=flow_m if USE_FLOW_PATHS else None,
    id_col=LAKE_ID_COL,
    flow_id_col=FLOW_LAKE_ID_COL
)

print("Exposure polygons rows:", len(exposure_polys))
exposure_polys.head()


  0%|          | 0/4519 [00:00<?, ?it/s]

Exposure polygons rows: 13557


,lake_id,distance_km,geometry
0,20,5,"POLYGON ((723619.972 3899215.819, 723595.896 3..."
1,20,10,"POLYGON ((728619.972 3899215.819, 728571.82 38..."
2,20,20,"POLYGON ((738619.972 3899215.819, 738523.667 3..."
3,31,5,"POLYGON ((717500.993 3903980.581, 717476.916 3..."
4,31,10,"POLYGON ((722500.993 3903980.581, 722452.84 39..."


### Optional: export exposure polygons (for QA/QC in QGIS)

In [ ]:
EXPORT_EXPOSURE_POLYS = True
OUT_GPKG = "pdgl_exposure_polygons.gpkg"

if EXPORT_EXPOSURE_POLYS:
    exposure_polys.to_file(OUT_GPKG, layer="exposure_corridors", driver="GPKG")
    print("Wrote:", OUT_GPKG)


## 5) Reproject population raster to the metric CRS
WorldPop is often in EPSG:4326 or another CRS. We'll warp it to the same metric CRS so area/distance buffering aligns.

This uses an in-memory GeoTIFF; for very large rasters you can adapt it to write a temporary file on disk.

In [6]:
def warp_raster_to_crs(src_path, dst_crs):
    with rasterio.open(src_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )
        kwargs = src.meta.copy()
        kwargs.update({
            "crs": dst_crs,
            "transform": transform,
            "width": width,
            "height": height
        })

        memfile = MemoryFile()
        with memfile.open(**kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.nearest
                )
        return memfile

pop_mem = warp_raster_to_crs(POP_RASTER_PATH, metric_crs)
pop_ds = pop_mem.open()

print("Pop raster CRS:", pop_ds.crs)
print("Pop raster shape:", (pop_ds.height, pop_ds.width))
print("Pixel size (approx):", pop_ds.transform.a, pop_ds.transform.e)


Pop raster CRS: EPSG:32643
Pop raster shape: (86450, 36219)
Pixel size (approx): 457.6202846132976 -457.6202846132976


## 6) Zonal sum: population within each exposure polygon
We compute the sum of raster values within each polygon. Assumes raster values represent people per pixel (as in typical WorldPop totals).

Performance tips:
- If you only need **totals across all PDGLs**, dissolve polygons per distance band and do 3 zonal sums.
- If you need **per-lake** results, we loop per polygon.

In [7]:
def pop_sum_in_polygon(dataset, polygon, all_touched=True):
    """Sum population raster values inside polygon."""
    if polygon is None or polygon.is_empty:
        return 0.0
    geom = [mapping(polygon)]
    out_img, out_transform = mask(dataset, geom, crop=True, all_touched=all_touched, filled=False)
    arr = out_img[0].astype("float64")
    # mask() returns masked values as 0 if filled=True; we used filled=False so it returns a MaskedArray
    if np.ma.isMaskedArray(arr):
        return float(arr.filled(0).sum())
    return float(np.nansum(arr))

# --- Option A: per-lake, per-band ---
rows = []
for _, r in tqdm(exposure_polys.iterrows(), total=len(exposure_polys)):
    lid = r[LAKE_ID_COL]
    d_km = r["distance_km"]
    poly = r.geometry
    s = pop_sum_in_polygon(pop_ds, poly, all_touched=True)
    rows.append({"lake_id": lid, "distance_km": d_km, "pop_sum": s})

per_lake = pd.DataFrame(rows)
per_lake.head()


  0%|          | 0/13557 [00:00<?, ?it/s]

,lake_id,distance_km,pop_sum
0,20,5,4.249511
1,20,10,23.335249
2,20,20,2538.102802
3,31,5,0.000000
4,31,10,1.265777


In [8]:
# Pivot to columns: pop_within_5km, pop_within_10km, ...
per_lake_pivot = (per_lake
                  .pivot_table(index="lake_id", columns="distance_km", values="pop_sum", aggfunc="sum")
                  .reset_index())

# Rename columns nicely
per_lake_pivot.columns = ["lake_id"] + [f"pop_within_{int(c)}km" for c in per_lake_pivot.columns[1:]]
per_lake_pivot.head()


,lake_id,pop_within_5km,pop_within_10km,pop_within_20km
0,20,4.249511,23.335249,2538.102802
1,31,0.000000,1.265777,2651.619597
2,32,0.000000,2.836739,1925.231189
3,34,0.000000,1.265777,2654.206027
4,35,0.000000,1.265777,2689.456621


In [9]:
# Join back to PDGL attributes (optional)
pdgl_cols_to_keep = [LAKE_ID_COL, PDGL_COL] + [
    c for c in pdgl.columns
    if c not in ["geometry", LAKE_ID_COL, PDGL_COL]
][:5]
# or: pdgl_cols_to_keep = list(dict.fromkeys(pdgl_cols_to_keep))
pdgl_attr = pdgl[pdgl_cols_to_keep].copy()
result_per_lake = pdgl_attr.merge(per_lake_pivot, left_on=LAKE_ID_COL, right_on="lake_id", how="left")

#result_per_lake.drop(columns=["lake_id"], inplace=True, errors="ignore")

result_per_lake.head()


,lake_id,is_PDGL,source_img,area_m2,area_km2,perimeter_km,parent_glacier_idx,pop_within_5km,pop_within_10km,pop_within_20km
0,20,True,20250902_060832_78_24da_3B_Visual_clip_reproje...,137.367649,0.000137,0.046211,356880.0,4.249511,23.335249,2538.102802
1,31,True,20250902_060832_78_24da_3B_Visual_clip_reproje...,131.741643,0.000132,0.045587,363571.0,0.000000,1.265777,2651.619597
2,32,True,20250902_060832_78_24da_3B_Visual_clip_reproje...,126.603402,0.000127,0.042660,329001.0,0.000000,2.836739,1925.231189
3,34,True,20250902_060832_78_24da_3B_Visual_clip_reproje...,64.097490,0.000064,0.030189,363545.0,0.000000,1.265777,2654.206027
4,35,True,20250902_060832_78_24da_3B_Visual_clip_reproje...,568.796972,0.000569,0.089243,664724.0,0.000000,1.265777,2689.456621


### Totals across all PDGLs (summed)
This gives the total population within 5/10/20 km downstream across all PDGLs.

In [10]:
totals = {}
for d in DISTANCE_BANDS_KM:
    col = f"pop_within_{int(d)}km"
    totals[col] = float(result_per_lake[col].fillna(0).sum())

totals_df = pd.DataFrame([totals])
totals_df


,pop_within_5km,pop_within_10km,pop_within_20km
0,1.581525e+07,7.778990e+07,4.149862e+08


## 7) Optional: Faster totals via dissolved polygons
If you only need totals (not per-lake), dissolve corridors by distance band and run 3 zonal sums.

In [ ]:
DO_FAST_TOTALS = True

if DO_FAST_TOTALS:
    dissolved = exposure_polys.dissolve(by="distance_km")
    fast_totals = []
    for d_km, row in dissolved.iterrows():
        poly = row.geometry
        s = pop_sum_in_polygon(pop_ds, poly, all_touched=True)
        fast_totals.append({"distance_km": d_km, "pop_sum_all_pdgls": s})
    fast_totals_df = pd.DataFrame(fast_totals).sort_values("distance_km")
    fast_totals_df


## 8) Save outputs

In [ ]:
OUT_PER_LAKE_CSV = "pdgl_population_exposure_per_lake.csv"
OUT_TOTALS_CSV = "pdgl_population_exposure_totals.csv"

result_per_lake.to_csv(OUT_PER_LAKE_CSV, index=False)
totals_df.to_csv(OUT_TOTALS_CSV, index=False)

print("Wrote:", OUT_PER_LAKE_CSV)
print("Wrote:", OUT_TOTALS_CSV)


## Notes / common pitfalls
- **Flow line direction:** ensure each flow line starts at the lake/outlet and goes downstream.
- **Distance meaning:** results are **cumulative** (within 5 km, within 10 km, within 20 km). If you want incremental rings (0–5, 5–10, 10–20), we can compute differences.
- **Population raster meaning:** WorldPop products can represent *people per pixel* for a specific year. Confirm your raster’s metadata.
- **No flow paths:** set `USE_FLOW_PATHS=False` to use circular buffers (quick but less accurate).